# LOGAN: feature distributions at fixed ORF length

**Draft caption.** Distributions of amino-acid (AA) entropy, predicted 3Di entropy, 12-state entropy and 3Di–12-state mutual information (MI) for complete ORFs of 100–149 aa (a–d) and 200–299 aa (e–h) in 50,009 LOGAN metagenomes. Classes: complete Prodigal genes (Prodigal+), and Prodigal− ORFs that are antisense shadows, same-strand frameshift shadows or intergenic. Densities from exact histograms over all 89.99 billion ORFs (0.01-bit bins; no sampling); n per class in the legends.

**Outputs:** `../images/figL01_feature_distributions.pdf` (vector) and `../images/figL01_feature_distributions.png` (300 dpi).

**Data** (plotting-ready, committed; produced by `../prep/logan_prepare_data.py` on Setonix):

- `../data/figL01_feature_distributions.tsv.gz` (9,391 rows; sha256 `581972d45375…`): complete ORFs; counts in 0.01-bit bins, summed over 90 shard histograms (all ORFs). Source on Setonix (`$LOGAN_ROOT`): 90 files under `analysis/v1/hist/` (each with sha256 in `logan_provenance.json`).

**Upstream.** LOGAN v1.2 metagenomes (50,009), genome_entropy #102 (closed by PR #115); analysis scripts `Validation/logan/scripts/` at commit `06080b4`. Full run provenance: `../data/logan_provenance.json`.

**Run:** `jupyter nbconvert --to notebook --execute figL01_feature_distributions.ipynb` from `manuscript/notebooks/`. Needs only pandas, numpy, matplotlib and seaborn; no HPC or remote paths.

In [ ]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

DATA, IMAGES = Path("../data"), Path("../images")
IMAGES.mkdir(exist_ok=True)
plt.rcParams["svg.hashsalt"] = "genome_entropy"   # deterministic SVG ids if SVG is ever written

# Colour slots (fixed order, colour-blind-checked categorical palette used across the LOGAN figures)
SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
sns.set_theme(context="paper", style="ticks", font_scale=1.15, rc={
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.facecolor": SURFACE, "figure.facecolor": "white", "grid.color": GRID,
    "axes.grid": True, "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "lines.linewidth": 2, "font.family": "DejaVu Sans"})


def panel(ax, letter, x=-0.08, y=1.04):
    ax.text(x, y, letter, transform=ax.transAxes, fontsize=15, fontweight="bold", va="bottom", ha="right")


def save(fig, name):
    """Vector PDF (fixed metadata, so reruns are identical) and a 300-dpi PNG preview."""
    fig.savefig(IMAGES / f"{name}.pdf", bbox_inches="tight", metadata={"CreationDate": None, "ModDate": None})
    fig.savefig(IMAGES / f"{name}.png", dpi=300, bbox_inches="tight", metadata={"Software": None})
    plt.close(fig)
    print("wrote", IMAGES / f"{name}.pdf", IMAGES / f"{name}.png")

In [ ]:
d = pd.read_csv(DATA / "figL01_feature_distributions.tsv.gz", sep="\t")
CLASS = {"prodigal_complete": "Prodigal+ (complete gene)", "antisense_overlap": "antisense shadow",
         "frameshift_overlap": "frameshift shadow", "intergenic": "intergenic"}
FEATS = [("aa_entropy", "AA entropy"), ("three_di_entropy", "3Di entropy"),
         ("twelve_state_entropy", "12-state entropy"), ("mi_3di_12st", "3Di–12st MI")]
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
letters = iter("abcdefgh")
for r, lb in enumerate(["100-149", "200-299"]):
    for c, (f, ftxt) in enumerate(FEATS):
        ax = axes[r, c]
        for i, (cl, ctxt) in enumerate(CLASS.items()):
            x = d[(d.length_bin_aa == lb) & (d.feature == f) & (d.orf_class == cl)].sort_values("bin_centre_bits")
            if x.empty:
                continue
            n = x["count"].sum()
            ax.plot(x.bin_centre_bits, x["count"] / n / 0.01, color=SLOTS[i], lw=1.8, label=f"{ctxt} (n={n:,})")
        ax.set_xlim(0, 2.0 if f == "mi_3di_12st" else 4.4)
        ax.set_xlabel(f"{ftxt} (bits)")
        ax.set_ylabel("density (per bit)" if c == 0 else "")
        if c == 0:
            ax.set_title(f"complete ORFs, {lb.replace('-', '–')} aa", loc="left", fontsize=11)
        if c == 3:
            ax.legend(fontsize=8, loc="upper right")
        panel(ax, next(letters))
fig.tight_layout()
save(fig, "figL01_feature_distributions")